In [1]:
import truststore
truststore.inject_into_ssl()

In [2]:
from dotenv import load_dotenv
load_dotenv()

True

Multi-representation Indexing

In [4]:
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

loader = WebBaseLoader("https://lilianweng.github.io/posts/2023-06-23-agent/")
docs = loader.load()

loader = WebBaseLoader("https://lilianweng.github.io/posts/2024-02-05-human-data-quality/")
docs.extend(loader.load())

In [5]:
import uuid

from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_groq import ChatGroq

chain = (
    {"doc": lambda x: x.page_content}
    | ChatPromptTemplate.from_template("Summarize the following document:\n\n{doc}")
    | ChatGroq(model="llama-3.3-70b-versatile", temperature=0)
    | StrOutputParser()
)

summaries = chain.batch(docs, {"max_concurrency": 5})

In [20]:
from langchain_core.stores import InMemoryStore
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_classic.retrievers import MultiVectorRetriever

# The vectorstore to use to index the child chunks
vectorstore = Chroma(collection_name="summaries",
                     embedding_function=HuggingFaceEmbeddings())

# The storage layer for the parent documents
store = InMemoryStore()
id_key = "doc_id"

# The retriever
retriever = MultiVectorRetriever(
    vectorstore=vectorstore,
    byte_store=store,
    id_key=id_key,
)
doc_ids = [str(uuid.uuid4()) for _ in docs]

# Docs linked to summaries
summary_docs = [
    Document(page_content=s, metadata={id_key: doc_ids[i]})
    for i, s in enumerate(summaries)
]

# Add
retriever.vectorstore.add_documents(summary_docs)
retriever.docstore.mset(list(zip(doc_ids, docs)))

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 8410.84it/s]


In [8]:
query = "Memory in agents"
sub_docs = vectorstore.similarity_search(query,k=1)
sub_docs[0]

Document(metadata={'doc_id': '3d9c034d-1a20-43fc-9172-1b7ffd8abe6b'}, page_content='The document discusses the concept of LLM (Large Language Model) powered autonomous agents, which are systems that use LLMs as their core controller. The author, Lilian Weng, explores the potential of LLMs to be used as a general problem solver, beyond just generating text.\n\nThe document is divided into three main components:\n\n1. **Planning**: This component involves breaking down complex tasks into smaller, manageable subgoals, and using self-reflection to refine past actions and correct mistakes. The author discusses various techniques, such as Chain of Thought (CoT) and Tree of Thoughts, that can be used to improve the planning capabilities of LLMs.\n2. **Memory**: This component involves the use of external memory stores to retain and recall information over extended periods. The author discusses the different types of memory, including short-term memory, long-term memory, and sensory memory, an

In [10]:
retrieved_docs = retriever.invoke(query,n_results=1)
retrieved_docs[0].page_content[0:500]

"\n\n\n\n\n\nLLM Powered Autonomous Agents | Lil'Log\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nLil'Log\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n|\n\n\n\n\n\n\nPosts\n\n\n\n\nArchive\n\n\n\n\nSearch\n\n\n\n\nTags\n\n\n\n\nFAQ\n\n\n\n\n\n\n\n\n\n      LLM Powered Autonomous Agents\n    \nDate: June 23, 2023  |  Estimated Reading Time: 31 min  |  Author: Lilian Weng\n\n\n \n\n\nTable of Contents\n\n\n\nAgent System Overview\n\nComponent One: Planning\n\nTask Decomposition\n\nSelf-Reflection\n\n\nComponent Two: Memory\n\nTypes of Memory\n\nMaximum Inner Product Search (MIPS)\n\n\nComponent Three:"

Raptor

In [ ]:
import numpy as np
from dataclasses import dataclass, field
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate

@dataclass
class RaptorNode:
    text: str               # original chunk ya summary
    embedding: np.ndarray
    level: int              # 0 = leaf, 1,2,... = higher summaries
    children: list = field(default_factory=list)  # child node indices

llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)
summary_chain = (
    {"doc": lambda x: x}
    | ChatPromptTemplate.from_template(
        "Concisely summarize the following text:\n\n{doc}"
    )
    | llm
    | StrOutputParser()
)

def summarize_cluster(texts: list[str]) -> str:
    combined = "\n\n".join(texts)
    return summary_chain.invoke(combined)

In [ ]:
import numpy as np
import umap
from sklearn.mixture import GaussianMixture

def embed_texts(documents: list[Document], model: HuggingFaceEmbeddings) -> np.ndarray:
    raw = [t.page_content for t in documents]
    return np.array(model.embed_documents(raw))

def get_optimal_clusters(embeddings: np.ndarray, max_k: int = 10) -> int:
    """BIC se best cluster count find karo"""
    bic_scores = []
    K = range(2, min(max_k, len(embeddings)))
    for k in K:
        gmm = GaussianMixture(n_components=k, random_state=42)
        gmm.fit(embeddings)
        bic_scores.append(gmm.bic(embeddings))
    return K[np.argmin(bic_scores)]  # lowest BIC = best fit

def cluster_texts(embeddings: np.ndarray, n_neighbors: int = 10) -> np.ndarray:
    """UMAP se reduce karo, phir GMM se cluster karo"""
    
    # Step 1: dimensionality reduce
    reduced = umap.UMAP(
        n_neighbors=n_neighbors,
        n_components=10,        # 768-dim → 10-dim
        min_dist=0.0,
        metric="cosine",
        random_state=42
    ).fit_transform(embeddings)
    
    # Step 2: optimal k find karo
    n_clusters = get_optimal_clusters(reduced)
    
    # Step 3: GMM fit karo (soft assignment)
    gmm = GaussianMixture(n_components=n_clusters, random_state=42)
    gmm.fit(reduced)
    
    # threshold = 0.5 → ek chunk multiple clusters mein ho sakta hai
    probs = gmm.predict_proba(reduced)
    labels = [np.where(p > 0.5)[0] for p in probs]  
    return labels

In [ ]:
def build_raptor_tree(
    docs: list[Document],
    embed_model: HuggingFaceEmbeddings,
    max_levels: int = 3
) -> list[RaptorNode]:
    
    all_nodes = []
    
    # Level 0: leaf nodes (original chunks)
    embeddings = embed_texts(docs, embed_model)

    for i, (doc, emb) in enumerate(zip(docs, embeddings)):
        text = doc.page_content
        all_nodes.append(RaptorNode(text=text, embedding=emb, level=0))
    
    current_level_nodes = all_nodes.copy()
    current_level = 0
    
    while current_level < max_levels and len(current_level_nodes) > 1:
        
        current_embeddings = np.array([n.embedding for n in current_level_nodes])
        
        # Cluster karo
        cluster_labels = cluster_texts(current_embeddings)
        
        # Clusters build karo (soft assignment handle karo)
        clusters: dict[int, list] = {}
        for node_idx, label_list in enumerate(cluster_labels):
            for cluster_id in label_list:
                clusters.setdefault(cluster_id, []).append(node_idx)
        
        # Har cluster ka summary banao
        next_level_nodes = []
        for cluster_id, node_indices in clusters.items():
            cluster_texts_list = [current_level_nodes[i].text for i in node_indices]
            summary_text = summarize_cluster(cluster_texts_list)
            summary_emb = embed_texts([summary_text], embed_model)[0]
            
            new_node = RaptorNode(
                text=summary_text,
                embedding=summary_emb,
                level=current_level + 1,
                children=[current_level_nodes[i] for i in node_indices]
            )
            next_level_nodes.append(new_node)
            all_nodes.append(new_node)
        
        current_level_nodes = next_level_nodes
        current_level += 1
        print(f"Level {current_level}: {len(next_level_nodes)} nodes")
    
    return all_nodes  # saare levels ke nodes

In [38]:
from langchain_community.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

def build_raptor_retriever(all_nodes: list[RaptorNode]):
    """
    Collapsed tree approach:
    saare nodes (sab levels) ek flat vectorstore mein
    """
    texts = [node.text for node in all_nodes]
    metadatas = [{"level": node.level} for node in all_nodes]
    
    vectorstore = Chroma.from_texts(
        texts=texts,
        embedding=HuggingFaceEmbeddings(),
        metadatas=metadatas,
        collection_name="raptor_tree"
    )
    return vectorstore

def raptor_retrieve(query: str, vectorstore, k: int = 5):
    """
    Query ke against collapsed tree search karo
    Level filter optional hai
    """
    results = vectorstore.similarity_search_with_score(query, k=k)
    
    for doc, score in results:
        level = doc.metadata["level"]
        label = "leaf" if level == 0 else f"summary (L{level})"
        print(f"[{label}] score={score:.3f} | {doc.page_content[:100]}...")
    
    return [doc for doc, _ in results]

In [ ]:
import bs4
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

loader = WebBaseLoader(
    web_path=("https://lilianweng.github.io/posts/2023-06-23-agent/",),
    bs_kwargs=dict(
        parse_only=bs4.SoupStrainer(
            class_=("post-content", "post-header", "post-title")
        )
    )
)

doc = loader.load()
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = splitter.split_documents(doc)

# 2. Tree building
embed_model = HuggingFaceEmbeddings()
all_nodes = build_raptor_tree(chunks, embed_model, max_levels=3)
# Output:
# Level 1: 6 nodes
# Level 2: 2 nodes  
# Level 3: 1 node (root)

# 3. Vectorstore mein index karo (collapsed tree)
vectorstore = build_raptor_retriever(all_nodes)

# 4. Query karo
query = "What is the main argument of this paper?"
results = raptor_retrieve(query, vectorstore, k=4)
# [summary (L2)] score=0.87 | The paper argues that...
# [leaf]         score=0.82 | Specifically in section 3...
# [summary (L1)] score=0.79 | The authors discuss...

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 7885.30it/s]


[[ 0.01402716  0.00798774  0.0054893  ...  0.05307384 -0.0330104
  -0.03781594]
 [ 0.02513788  0.00249718 -0.04042218 ...  0.01105731 -0.07675707
  -0.03820436]
 [ 0.02547769 -0.04406551 -0.02262818 ...  0.01163945  0.0170565
  -0.02050594]
 ...
 [ 0.03323341  0.00585365 -0.02542839 ...  0.05726517 -0.08818464
  -0.01747221]
 [ 0.01904619  0.02766336 -0.02253631 ...  0.03831076 -0.09786347
  -0.05331308]
 [-0.02406973  0.02388948 -0.06167518 ...  0.05868635 -0.00572989
  -0.07039834]]


SyntaxError: 'return' outside function (2973688870.py, line 21)

ColBERT

In [3]:
from ragatouille import RAGPretrainedModel
RAG = RAGPretrainedModel.from_pretrained("colbert-ir/colbertv2.0")

/var/folders/_d/tj_f_hcs5gd3hx64vjm05drw0000gp/T/ipykernel_24837/276443619.py:1: UserWarning: 
********************************************************************************
RAGatouille WARNING: Future Release Notice
--------------------------------------------
RAGatouille version 0.0.10 will be migrating to a PyLate backend 
instead of the current Stanford ColBERT backend.
PyLate is a fully mature, feature-equivalent backend, that greatly facilitates compatibility.
However, please pin version <0.0.10 if you require the Stanford ColBERT backend.
********************************************************************************
  from ragatouille import RAGPretrainedModel
/Users/bharat.goyal1/rag/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[May 27, 12:46:37] Loading segmented_maxsim_cpp extension (set COLBERT_LOAD_TORCH_EXTENSION_VERBOSE=True for more info)...


/Users/bharat.goyal1/rag/.venv/lib/python3.11/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
/Users/bharat.goyal1/rag/.venv/lib/python3.11/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
/Users/bharat.goyal1/rag/.venv/lib/python3.11/site-packages/colbert/utils/amp.py:12: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradSc

In [4]:
import requests

def get_wikipedia_page(title: str):
    """
    Retrieve the full text content of a Wikipedia page.

    :param title: str - Title of the Wikipedia page.
    :return: str - Full text content of the page as raw string.
    """
    # Wikipedia API endpoint
    URL = "https://en.wikipedia.org/w/api.php"

    # Parameters for the API request
    params = {
        "action": "query",
        "format": "json",
        "titles": title,
        "prop": "extracts",
        "explaintext": True,
    }

    # Custom User-Agent header to comply with Wikipedia's best practices
    headers = {"User-Agent": "RAGatouille_tutorial/0.0.1 (ben@clavie.eu)"}

    response = requests.get(URL, params=params, headers=headers)
    data = response.json()

    # Extracting page content
    page = next(iter(data["query"]["pages"].values()))
    return page["extract"] if "extract" in page else None

full_document = get_wikipedia_page("Hayao_Miyazaki")

In [5]:
RAG.index(
    collection=[full_document],
    index_name="Miyazaki-123",
    max_document_length=180,
    split_documents=True,
)

---- WARNING! You are using PLAID with an experimental replacement for FAISS for greater compatibility ----
This is a behaviour change from RAGatouille 0.8.0 onwards.
This works fine for most users and smallish datasets, but can be considerably slower than FAISS and could cause worse results in some situations.
If you're confident with FAISS working on your machine, pass use_faiss=True to revert to the FAISS-using behaviour.
--------------------


[May 27, 12:46:58] #> Creating directory .ragatouille/colbert/indexes/Miyazaki-123 


[May 27, 12:47:03] [0] 		 #> Encoding 124 passages..


  0%|          | 0/4 [00:00<?, ?it/s]/Users/bharat.goyal1/rag/.venv/lib/python3.11/site-packages/colbert/utils/amp.py:15: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  return torch.cuda.amp.autocast() if self.activated else NullContextManager()
/Users/bharat.goyal1/rag/.venv/lib/python3.11/site-packages/torch/cuda/amp/autocast_mode.py:54: UserWarning: CUDA is not available or torch_xla is imported. Disabling autocast.
  super().__init__(
 25%|██▌       | 1/4 [00:02<00:08,  2.81s/it]/Users/bharat.goyal1/rag/.venv/lib/python3.11/site-packages/colbert/utils/amp.py:15: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  return torch.cuda.amp.autocast() if self.activated else NullContextManager()
/Users/bharat.goyal1/rag/.venv/lib/python3.11/site-packages/torch/cuda/amp/autocast_mode.py:54: UserWarning: CUDA is not available or torch_xla is 

[May 27, 12:47:09] [0] 		 avg_doclen_est = 131.9596710205078 	 len(local_sample) = 124
[May 27, 12:47:09] [0] 		 Creating 1,024 partitions.
[May 27, 12:47:09] [0] 		 *Estimated* 16,362 embeddings.
[May 27, 12:47:09] [0] 		 #> Saving the indexing plan to .ragatouille/colbert/indexes/Miyazaki-123/plan.json ..


used 20 iterations (0.5693s) to cluster 15545 items into 1024 clusters
[0.037, 0.042, 0.038, 0.034, 0.033, 0.038, 0.033, 0.037, 0.034, 0.035, 0.034, 0.037, 0.033, 0.037, 0.036, 0.038, 0.033, 0.033, 0.034, 0.038, 0.037, 0.033, 0.034, 0.036, 0.037, 0.032, 0.038, 0.034, 0.034, 0.036, 0.035, 0.038, 0.038, 0.035, 0.034, 0.034, 0.036, 0.033, 0.037, 0.039, 0.036, 0.038, 0.034, 0.032, 0.035, 0.032, 0.033, 0.039, 0.037, 0.033, 0.034, 0.036, 0.035, 0.035, 0.035, 0.035, 0.039, 0.038, 0.041, 0.032, 0.034, 0.037, 0.035, 0.035, 0.037, 0.038, 0.035, 0.037, 0.032, 0.033, 0.035, 0.034, 0.032, 0.037, 0.036, 0.035, 0.034, 0.039, 0.033, 0.035, 0.036, 0.037, 0.033, 0.04, 0.034, 0.037, 0.037, 0.036, 0.032, 0.043, 0.036, 0.036, 0.035, 0.036, 0.035, 0.035, 0.036, 0.032, 0.037, 0.036, 0.039, 0.041, 0.039, 0.036, 0.038, 0.036, 0.035, 0.033, 0.038, 0.034, 0.036, 0.035, 0.034, 0.031, 0.036, 0.037, 0.036, 0.034, 0.037, 0.039, 0.032, 0.033, 0.034, 0.037, 0.032, 0.037, 0.035, 0.037]


0it [00:00, ?it/s]

...

100%|██████████| 4/4 [00:03<00:00,  1.03it/s]
1it [00:03,  3.94s/it]
100%|██████████| 1/1 [00:00<00:00, 1267.54it/s]

[May 27, 12:47:13] #> Optimizing IVF to store map from centroids to list of pids..
[May 27, 12:47:13] #> Building the emb2pid mapping..
[May 27, 12:47:13] len(emb2pid) = 16363



100%|██████████| 1024/1024 [00:00<00:00, 228967.23it/s]

[May 27, 12:47:13] #> Saved optimized IVF to .ragatouille/colbert/indexes/Miyazaki-123/ivf.pid.pt
Done indexing!


'.ragatouille/colbert/indexes/Miyazaki-123'

In [6]:
results = RAG.search(query="What animation studio did Miyazaki found?", k=3)
results

Loading searcher for index Miyazaki-123 for the first time... This may take a few seconds


/Users/bharat.goyal1/rag/.venv/lib/python3.11/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


[May 27, 12:47:31] #> Loading codec...
[May 27, 12:47:31] #> Loading IVF...
[May 27, 12:47:31] Loading segmented_lookup_cpp extension (set COLBERT_LOAD_TORCH_EXTENSION_VERBOSE=True for more info)...


/Users/bharat.goyal1/rag/.venv/lib/python3.11/site-packages/colbert/utils/amp.py:12: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler()
/Users/bharat.goyal1/rag/.venv/lib/python3.11/site-packages/torch/cuda/amp/grad_scaler.py:31: UserWarning: torch.cuda.amp.GradScaler is enabled, but CUDA is not available.  Disabling.
  super().__init__(


[May 27, 12:47:38] #> Loading doclens...


...


100%|██████████| 1/1 [00:00<00:00, 263.93it/s]

[May 27, 12:47:38] Loading filter_pids_cpp extension (set COLBERT_LOAD_TORCH_EXTENSION_VERBOSE=True for more info)...


[May 27, 12:47:45] Loading decompress_residuals_cpp extension (set COLBERT_LOAD_TORCH_EXTENSION_VERBOSE=True for more info)...
Searcher loaded!

#> QueryTokenizer.tensorize(batch_text[0], batch_background[0], bsize) ==
#> Input: What animation studio did Miyazaki found?, 		 True, 		 None
#> Output IDs: torch.Size([32]), tensor([  101,     1,  2054,  7284,  2996,  2106,  2771,  3148, 18637,  2179,
         1029,   102,   103,   103,   103,   103,   103,   103,   103,   103,
          103,   103,   103,   103,   103,   103,   103,   103,   103,   103,
          103,   103])
#> Output Mask: torch.Size([32]), tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0])



/Users/bharat.goyal1/rag/.venv/lib/python3.11/site-packages/colbert/utils/amp.py:15: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  return torch.cuda.amp.autocast() if self.activated else NullContextManager()
/Users/bharat.goyal1/rag/.venv/lib/python3.11/site-packages/torch/cuda/amp/autocast_mode.py:54: UserWarning: CUDA is not available or torch_xla is imported. Disabling autocast.
  super().__init__(


[{'content': '=== Studio Ghibli ===\n\n\n==== Foundation and Laputa (1985–1987) ====\n\nFollowing the success of Nausicaä of the Valley of the Wind, Miyazaki and Takahata founded the animation production company Studio Ghibli on June 15, 1985, as a subsidiary of Tokuma Shoten, with offices in Kichijōji designed by Miyazaki. The studio\'s name had been registered a year earlier; Miyazaki named it after the nickname of the Caproni Ca.309 aircraft, meaning "a hot wind that blows in the desert" in Italian.',
  'score': 25.730194091796875,
  'rank': 1,
  'document_id': 'f948e31c-49c5-414e-ae21-b50172c8a4df',
  'passage_id': 42},
 {'content': 'Hayao Miyazaki (宮崎 駿 or 宮﨑 駿, Miyazaki Hayao; [mijaꜜzaki hajao]; born January 5, 1941) is a Japanese animator, filmmaker, and manga artist. He co-founded Studio Ghibli and serves as its honorary chairman. Throughout his career, Miyazaki has attained international acclaim as a masterful storyteller and creator of Japanese animated feature films, and is 

In [7]:
retriever = RAG.as_langchain_retriever(k=3)
retriever.invoke("What animation studio did Miyazaki found?")

/Users/bharat.goyal1/rag/.venv/lib/python3.11/site-packages/colbert/utils/amp.py:15: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  return torch.cuda.amp.autocast() if self.activated else NullContextManager()
/Users/bharat.goyal1/rag/.venv/lib/python3.11/site-packages/torch/cuda/amp/autocast_mode.py:54: UserWarning: CUDA is not available or torch_xla is imported. Disabling autocast.
  super().__init__(


[Document(metadata={}, page_content='=== Studio Ghibli ===\n\n\n==== Foundation and Laputa (1985–1987) ====\n\nFollowing the success of Nausicaä of the Valley of the Wind, Miyazaki and Takahata founded the animation production company Studio Ghibli on June 15, 1985, as a subsidiary of Tokuma Shoten, with offices in Kichijōji designed by Miyazaki. The studio\'s name had been registered a year earlier; Miyazaki named it after the nickname of the Caproni Ca.309 aircraft, meaning "a hot wind that blows in the desert" in Italian.'),
 Document(metadata={}, page_content='Hayao Miyazaki (宮崎 駿 or 宮﨑 駿, Miyazaki Hayao; [mijaꜜzaki hajao]; born January 5, 1941) is a Japanese animator, filmmaker, and manga artist. He co-founded Studio Ghibli and serves as its honorary chairman. Throughout his career, Miyazaki has attained international acclaim as a masterful storyteller and creator of Japanese animated feature films, and is widely regarded as one of the most influential and accomplished filmmakers 